# Convert PDFs to Markdown

This notebook converts all `.pdf` files under the workspace root into `.md` files for reference.
It extracts text where possible and writes clear limitation notes when extraction is not possible.

In [1]:
from pathlib import Path
import re
import subprocess
import sys

ROOT = Path('/workspaces/Trading-Web')
MIRROR_STRUCTURE = False  # False -> output beside source PDFs
OVERWRITE = True

# Install extraction backend if needed.
try:
    import fitz  # PyMuPDF
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pymupdf'])
    import fitz


def normalize_stem(stem: str) -> str:
    # Deterministic and filesystem-safe naming helper.
    stem = re.sub(r'\s+', ' ', stem).strip()
    return re.sub(r'[^A-Za-z0-9 _\-().]', '', stem)


def discover_pdfs(root: Path):
    return sorted(root.rglob('*.pdf'))


def md_target_for(pdf_path: Path, root: Path) -> Path:
    if MIRROR_STRUCTURE:
        rel = pdf_path.relative_to(root)
        out = root / 'pdf_markdown' / rel.with_suffix('.md')
    else:
        safe_stem = normalize_stem(pdf_path.stem)
        out = pdf_path.with_name(f'{safe_stem}.md')
    out.parent.mkdir(parents=True, exist_ok=True)
    return out


def extract_pdf_text(pdf_path: Path):
    lines = []
    with fitz.open(pdf_path) as doc:
        lines.append(f'# {pdf_path.stem}')
        lines.append('')
        lines.append(f'**Source PDF:** {pdf_path.name}')
        lines.append(f'**Pages:** {doc.page_count}')
        lines.append('')
        any_text = False
        for i, page in enumerate(doc, start=1):
            text = page.get_text('text').strip()
            lines.append(f'## Page {i}')
            lines.append('')
            if text:
                any_text = True
                lines.append(text)
            else:
                lines.append('_No extractable text on this page (possibly image-only)._')
            lines.append('')
    return '\n'.join(lines), any_text


def convert_one(pdf_path: Path, root: Path):
    out = md_target_for(pdf_path, root)
    if out.exists() and not OVERWRITE:
        return {'pdf': pdf_path, 'md': out, 'status': 'skipped', 'error': ''}
    try:
        md_text, any_text = extract_pdf_text(pdf_path)
        if not any_text:
            md_text += '\n## Note\n\nThis PDF appears to be image-based or text extraction is unavailable.\n'
        out.write_text(md_text, encoding='utf-8')
        if out.stat().st_size == 0:
            raise ValueError('Generated markdown is empty')
        return {'pdf': pdf_path, 'md': out, 'status': 'converted', 'error': ''}
    except Exception as exc:
        fallback = [
            f'# {pdf_path.stem}',
            '',
            f'**Source PDF:** {pdf_path.name}',
            '',
            '## Extraction Limitation',
            '',
            f'Could not extract text automatically: `{type(exc).__name__}: {exc}`',
            '',
            'Try OCR if this is a scanned/image PDF.',
            ''
        ]
        out.write_text('\n'.join(fallback), encoding='utf-8')
        return {'pdf': pdf_path, 'md': out, 'status': 'failed-fallback-written', 'error': str(exc)}


pdf_files = discover_pdfs(ROOT)
print(f'Found {len(pdf_files)} PDF files')
for p in pdf_files:
    print('-', p.relative_to(ROOT))

results = [convert_one(p, ROOT) for p in pdf_files]

converted = [r for r in results if r['status'] == 'converted']
skipped = [r for r in results if r['status'] == 'skipped']
failed = [r for r in results if r['status'] == 'failed-fallback-written']

print('\nSummary')
print('Converted:', len(converted))
print('Skipped:', len(skipped))
print('Failed with fallback written:', len(failed))

# Validation: verify output exists and is non-empty.
invalid = []
for r in results:
    md = r['md']
    if (not md.exists()) or (md.stat().st_size == 0):
        invalid.append(md)

print('Invalid outputs:', len(invalid))
if invalid:
    for i in invalid:
        print('-', i)

print('\nGenerated Markdown files:')
for r in results:
    print('-', r['md'].relative_to(ROOT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 36.6 MB/s  0:00:00m0:00:010:01



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Found 2 PDF files
- FOREX_MILLIONAIRE_IN_365_DAYS_BY_GODS_GRACE_by_Louis_Jr_Tshakoane (2).pdf
- TRENDLINE TRADING STRATEGY (1)(1) (3).pdf

Summary
Converted: 2
Skipped: 0
Failed with fallback written: 0
Invalid outputs: 0

Generated Markdown files:
- FOREX_MILLIONAIRE_IN_365_DAYS_BY_GODS_GRACE_by_Louis_Jr_Tshakoane (2).md
- TRENDLINE TRADING STRATEGY (1)(1) (3).md
